# Local defect research in Colab
Select a GPU runtime. Supply your own permitted competition ZIP and recovery directory. No data, credentials or private Drive IDs are embedded. GPU performance and cloud tracking have not yet been verified. Run one measured baseline before a tuning plan.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import torch
import torchvision
import numpy
import PIL
import sklearn
from pathlib import Path
import subprocess
import sys
import time
import shutil
REPO_URL = "https://github.com/rlaqudwn1/dku-defect-detection-2026.git" # replace with the actual public repository URL if different
REPO = Path("/content/defectlab")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
%cd /content/defectlab
print({"existing_torch": torch.__version__, "existing_torchvision": torchvision.__version__, "numpy": numpy.__version__, "Pillow": PIL.__version__, "sklearn": sklearn.__version__})
# Keep Colab GPU wheels. Install only the thin package; inspect missing libraries explicitly.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=open("/content/environment-colab.txt", "w"), check=True)


Set paths before continuing. ZIP must contain open/train.csv and open/test/. Keep secrets in Colab Secrets or environment variables. W&B is disabled by default. A recovery copy must be verified after copying; mount availability alone is not durable evidence.

In [ ]:
import torch
ZIP_PATH = Path("/content/drive/MyDrive/competition/open.zip")
RECOVERY = Path("/content/drive/MyDrive/competition/recovery")
DATA_ROOT = Path("/content/competition/open")
assert ZIP_PATH.is_file(), "Set ZIP_PATH to your private dataset ZIP"
started = time.perf_counter()
shutil.copy2(ZIP_PATH, "/content/open.zip")
subprocess.run([sys.executable, "-m", "zipfile", "-e", "/content/open.zip", "/content/competition"], check=True)
print({"prepare_seconds": time.perf_counter()-started})
print({"python":sys.version,"torch":torch.__version__,"cuda":torch.cuda.is_available(),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"quota":"unknown"})
assert torch.cuda.is_available(), "Select GPU runtime or explicitly create a CPU config"


In [ ]:
def run(*args):
    subprocess.run([sys.executable,"-m","defectlab",*map(str,args)],check=True)
run("verify", DATA_ROOT) # add --manifest and your private supplier manifest when available
run("profile", DATA_ROOT, "runs/eda")
run("split", "runs/eda/profile.json", "runs/split.json")
run("train", DATA_ROOT, "runs/split.json", "configs/baseline.toml", "runs/baseline")
run("evaluate", DATA_ROOT, "runs/split.json", "runs/baseline/best.pt", "runs/baseline/validation.json")
run("predict", DATA_ROOT, DATA_ROOT/"sample_submission.csv", "runs/baseline/best.pt", "runs/baseline/submission.csv")


In [ ]:
from defectlab.common import hash_file
RECOVERY.mkdir(parents=True,exist_ok=True)
for source in [Path("runs/split.json"), *Path("runs/baseline").glob("*")]:
    if source.is_file():
        target=RECOVERY/source.name
        shutil.copy2(source,target)
        assert hash_file(source)==hash_file(target)
print("Recovery files copied and locally re-read; verify downloading in a new session separately.")


After a new session: install the same code/environment, copy split/checkpoints/config from your recovery directory, verify hashes and run evaluate/predict. Resume with `--resume runs/baseline/last.pt`; increase epochs only if extending the run. Compare validation files with `reproduce` using a tolerance fixed before final selection. Record failures, actual GPU model/time and unknown session tails. This notebook does not submit to the competition or purchase GPU time.